# ⚡ Building a ReAct Agent

**Day 10 — Notebook 4 of 5 | Estimated Time: 40 minutes**

---

## What You'll Learn
- Implement a full ReAct agent class with configurable tools, memory, and stopping criteria
- Add conversation memory to support multi-turn interactions
- Implement step-level logging for observability
- Handle tool errors and agent failures gracefully
- Build a research assistant agent with search, calculation, and date tools

---

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0"

In [ ]:
import sys, os, json, datetime, sqlite3
from dataclasses import dataclass, field
from typing import Any, Callable

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types

client = genai.Client(api_key=get_api_key())
MODEL = "gemini-2.5-flash"
print("✅ Ready!")

---

## 1. The ReAct Agent Class

In [ ]:
@dataclass
class AgentStep:
    """Records a single step in the agent's execution trace."""
    step_num: int
    action: str            # tool name called, or "final_answer"
    args: dict
    result: Any
    timestamp: str = field(default_factory=lambda: datetime.datetime.utcnow().isoformat())


class ReActAgent:
    """
    A ReAct agent with:
    - Configurable tools and registry
    - Conversation history (multi-turn)
    - Step-level execution trace
    - Graceful error handling
    - Configurable max steps and verbosity
    """

    def __init__(
        self,
        tools: list,
        tool_registry: dict[str, Callable],
        system_prompt: str = "",
        model: str = MODEL,
        max_steps: int = 10,
        verbose: bool = True,
    ):
        self.tools = tools
        self.registry = tool_registry
        self.model = model
        self.max_steps = max_steps
        self.verbose = verbose
        self.system_prompt = system_prompt

        # Conversation history — grows across turns
        self.conversation_history: list[types.Content] = []

        # Execution trace for current task
        self.current_trace: list[AgentStep] = []

        # Stats
        self.total_tool_calls = 0
        self.total_turns = 0

        # Initialise with system prompt if provided
        if system_prompt:
            self.conversation_history.append(
                types.Content(role="user", parts=[types.Part(text=system_prompt)])
            )

    def ask(self, query: str) -> str:
        """Process a user query through the ReAct loop. Maintains conversation history."""
        self.total_turns += 1
        self.current_trace = []

        if self.verbose:
            print(f"\n{'━'*60}")
            print(f"Turn {self.total_turns}: {query}")
            print('━'*60)

        # Add user message to history
        self.conversation_history.append(
            types.Content(role="user", parts=[types.Part(text=query)])
        )

        for step_num in range(1, self.max_steps + 1):
            response = client.models.generate_content(
                model=self.model,
                contents=self.conversation_history,
                config=types.GenerateContentConfig(tools=self.tools),
            )

            part = response.candidates[0].content.parts[0]

            # ── Final answer ──────────────────────────────────────────
            if not hasattr(part, "function_call") or part.function_call is None:
                answer = part.text.strip()
                self.current_trace.append(
                    AgentStep(step_num, "final_answer", {}, answer)
                )
                # Add to conversation history
                self.conversation_history.append(
                    types.Content(role="model", parts=[types.Part(text=answer)])
                )
                if self.verbose:
                    print(f"[Answer] {answer}")
                return answer

            # ── Tool call ─────────────────────────────────────────────
            fn_call = part.function_call
            fn_name = fn_call.name
            fn_args = dict(fn_call.args)

            if self.verbose:
                print(f"[Step {step_num}] {fn_name}({json.dumps(fn_args, default=str)[:70]})")

            # Execute tool (catch exceptions)
            if fn_name not in self.registry:
                fn_result = {"error": f"Unknown tool: {fn_name}"}
            else:
                try:
                    fn_result = self.registry[fn_name](**fn_args)
                except TypeError as e:
                    fn_result = {"error": f"Bad arguments: {e}"}
                except Exception as e:
                    fn_result = {"error": f"Tool error: {e}"}

            if self.verbose:
                print(f"         → {str(fn_result)[:100]}")

            # Record step
            self.current_trace.append(AgentStep(step_num, fn_name, fn_args, fn_result))
            self.total_tool_calls += 1

            # Append to conversation
            self.conversation_history.append(
                types.Content(role="model", parts=[types.Part(function_call=fn_call)])
            )
            self.conversation_history.append(
                types.Content(role="user", parts=[types.Part(
                    function_response=types.FunctionResponse(name=fn_name, response=fn_result)
                )])
            )

        error_msg = f"Agent reached max steps ({self.max_steps}) without a final answer."
        if self.verbose:
            print(f"[Error] {error_msg}")
        return error_msg

    def reset_conversation(self):
        """Clear conversation history (but keep system prompt)."""
        if self.system_prompt:
            self.conversation_history = [
                types.Content(role="user", parts=[types.Part(text=self.system_prompt)])
            ]
        else:
            self.conversation_history = []
        self.total_turns = 0

    def print_trace(self):
        """Print the execution trace from the last task."""
        print(f"\nExecution trace ({len(self.current_trace)} steps):")
        for step in self.current_trace:
            if step.action == "final_answer":
                print(f"  [Step {step.step_num}] → Final answer")
            else:
                print(f"  [Step {step.step_num}] {step.action}")
                print(f"    Args:   {step.args}")
                print(f"    Result: {str(step.result)[:80]}")

    def stats(self) -> dict:
        return {
            "total_turns": self.total_turns,
            "total_tool_calls": self.total_tool_calls,
            "history_length": len(self.conversation_history),
        }


print("ReActAgent class defined.")

---

## 2. Build a Research Assistant Agent

In [ ]:
# --- Tools ---
NEWS_DB = [
    {"title": "AI Investment Hits $91B in 2023", "date": "2024-01-10",
     "snippet": "Global AI investment reached $91 billion in 2023. OpenAI raised $10B, Anthropic raised $7.3B. Enterprise AI adoption grew 35% year-on-year."},
    {"title": "Gemini 2.0 Released", "date": "2024-12-05",
     "snippet": "Google DeepMind released Gemini 2.0, featuring a 1M token context window, native tool use, and multimodal output capabilities."},
    {"title": "EU AI Act Signed Into Law", "date": "2024-07-12",
     "snippet": "The EU AI Act was signed into law in July 2024. High-risk AI systems must comply by 2026."},
    {"title": "RAG Remains Top Enterprise AI Pattern", "date": "2024-09-08",
     "snippet": "Gartner: RAG is used in 73% of production LLM applications as of Q3 2024."},
    {"title": "Python Tops Survey 2024", "date": "2024-06-15",
     "snippet": "Python is the most used language for the fourth year. 51% of developers use it regularly."},
]


def web_search(query: str, num_results: int = 3) -> dict:
    query_lower = query.lower()
    scored = []
    for a in NEWS_DB:
        score = sum(w in a["title"].lower() or w in a["snippet"].lower() for w in query_lower.split())
        if score > 0:
            scored.append((score, a))
    scored.sort(key=lambda x: x[0], reverse=True)
    results = [dict(a) for _, a in scored[:num_results]]
    return {"results": results} if results else {"results": [], "message": "No results"}


def calculate(expression: str) -> dict:
    try:
        allowed = set("0123456789.+-*/() ")
        if not all(c in allowed for c in expression):
            return {"error": "Only arithmetic allowed"}
        return {"result": round(eval(expression), 4)}  # noqa
    except Exception as e:
        return {"error": str(e)}


def get_current_date() -> dict:
    now = datetime.datetime.utcnow()
    return {"date": now.strftime("%Y-%m-%d"), "day_of_week": now.strftime("%A"), "year": now.year}


# --- Tool declarations ---
RESEARCH_TOOLS = [
    types.Tool(function_declarations=[
        types.FunctionDeclaration(
            name="web_search",
            description="Search for recent news about AI, technology, enterprise software, and related topics.",
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "query":       types.Schema(type=types.Type.STRING, description="Search query"),
                    "num_results": types.Schema(type=types.Type.INTEGER, description="Number of results (max 5)"),
                },
                required=["query"],
            ),
        ),
        types.FunctionDeclaration(
            name="calculate",
            description="Evaluate an arithmetic expression. Example: '91 * 1000000000 / 100'",
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={"expression": types.Schema(type=types.Type.STRING)},
                required=["expression"],
            ),
        ),
        types.FunctionDeclaration(
            name="get_current_date",
            description="Get the current date. Use when the query involves 'today', 'now', or 'current'.",
            parameters=types.Schema(type=types.Type.OBJECT, properties={}),
        ),
    ])
]

RESEARCH_REGISTRY = {
    "web_search": web_search,
    "calculate": calculate,
    "get_current_date": get_current_date,
}

RESEARCH_SYSTEM = """You are a research assistant. Use your tools to find accurate, 
up-to-date information. Always search before answering factual questions about 
recent events. Show your working when making calculations."""


agent = ReActAgent(
    tools=RESEARCH_TOOLS,
    tool_registry=RESEARCH_REGISTRY,
    system_prompt=RESEARCH_SYSTEM,
    max_steps=8,
    verbose=True,
)

print("Research agent created.")

In [ ]:
# Run a research task
answer = agent.ask(
    "What was the total AI investment in 2023? If that money were divided equally "
    "among the world's 8 billion people, how many dollars would each person get?"
)

agent.print_trace()

---

## 3. Multi-Turn Conversation

In [ ]:
# Reset and test multi-turn — the agent remembers previous answers
agent.reset_conversation()

turn_1 = agent.ask("What is the EU AI Act and when was it signed?")
print()

turn_2 = agent.ask("How many years from signing until high-risk systems must comply?")
print()

turn_3 = agent.ask("What year is that deadline?")

print(f"\nStats: {agent.stats()}")

---

## 4. Observability — Reading the Trace

In [ ]:
# Run a task and inspect the full trace
agent.reset_conversation()
agent.ask(
    "Search for the latest news about RAG and AI investment. "
    "Then tell me which story is more recent."
)

# Detailed trace analysis
print("\nFull trace analysis:")
tool_call_counts = {}
for step in agent.current_trace:
    if step.action != "final_answer":
        tool_call_counts[step.action] = tool_call_counts.get(step.action, 0) + 1

print(f"  Tool calls: {tool_call_counts}")
print(f"  Total steps: {len(agent.current_trace)}")
print(f"  Steps to answer: {next((s.step_num for s in agent.current_trace if s.action == 'final_answer'), 'N/A')}")

---

## 5. Handling Errors Gracefully

In [ ]:
# Test agent behaviour when a tool returns an error
agent.reset_conversation()

print("Testing error recovery:")
answer = agent.ask(
    "Search for news about quantum computing breakthroughs in 2024."
    # This query won't match any articles — agent should handle gracefully
)
print(f"\nFinal answer: {answer}")

print("\nTest division by zero recovery:")
answer = agent.ask("Calculate 100 divided by 0.")
print(f"Final answer: {answer}")

---

## 🏋️ Exercise 1: Add a Note-Taking Tool

Give the agent the ability to take notes during a long research task.

In [ ]:
# Exercise 1: Note-taking tool
# Long research tasks need scratch-pad notes so the agent can track
# what it has found without relying on context memory alone.
#
# TODO:
# 1. Create a Notepad class with:
#    - add_note(title, content) → store a research note
#    - get_notes() → return all notes as a list
#    - search_notes(keyword) → find notes containing a keyword
#
# 2. Create types.Tool declarations for these three functions
#
# 3. Add to RESEARCH_REGISTRY and RESEARCH_TOOLS
#
# 4. Test with a multi-step research task:
#    "Research the EU AI Act and AI investment levels.
#     Take notes on each topic separately, then write a brief 
#     summary connecting the two."

class Notepad:
    def __init__(self):
        self._notes = []

    def add_note(self, title: str, content: str) -> dict:
        # TODO: Implement
        pass

    def get_notes(self) -> dict:
        # TODO: Implement
        pass

    def search_notes(self, keyword: str) -> dict:
        # TODO: Implement
        pass


notepad = Notepad()
# TODO: Build tool declaration and add to agent

---

## 🏋️ Exercise 2: Conversation Export

After a conversation, export a human-readable transcript with metadata.

In [ ]:
# Exercise 2: Conversation export
# TODO:
# 1. Add an export_transcript(path) method to ReActAgent
#    Output format:
#    {
#      "exported_at": "...",
#      "stats": {...},
#      "turns": [
#        {
#          "turn": 1,
#          "user_query": "...",
#          "steps": [{"tool": ..., "args": ..., "result": ...}, ...],
#          "final_answer": "..."
#        }
#      ]
#    }
#
# 2. Run a 3-turn conversation, then export to /tmp/agent_transcript.json
# 3. Print the file contents nicely formatted
#
# Hint: you'll need to store per-turn trace data, not just current_trace

# TODO: Implement export_transcript and test it

---

## Key Takeaways

| Feature | Implementation |
|---------|---------------|
| Tool dispatch | `tool_registry` dict maps function name → callable |
| Conversation memory | `conversation_history` list — grows with each turn |
| Execution trace | `AgentStep` dataclass captures every action |
| Error handling | `try/except` in tool dispatch → error dict sent back to model |
| Multi-turn | Model references earlier turns automatically via history |
| Observability | `print_trace()` + `stats()` for debugging and monitoring |

---

## 📚 Further Reading

| Resource | Type | Description |
|----------|------|-------------|
| [LangChain AgentExecutor](https://python.langchain.com/docs/modules/agents/) | Docs | Production-grade agent executor |
| [LangGraph](https://langchain-ai.github.io/langgraph/) | Docs | Stateful multi-agent workflows |
| [Tracing with LangSmith](https://docs.smith.langchain.com/) | Docs | Observability for LLM agents |

---

**Next up:** [05_Multi_Agent_RAG_System.ipynb](./05_Multi_Agent_RAG_System.ipynb)